In [6]:
import os
from dotenv import load_dotenv

# Force load the .env file
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

token = os.environ.get("GEMINI_API_KEY")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"GEMINI_API_KEY loaded ({len(token)} characters): {masked}")
else:
    print("GEMINI_API_KEY not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'GEMINI_API_KEY', and load_dotenv() ran without error.")

GEMINI_API_KEY loaded (53 characters): AQ.A...EMNg


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

token = os.environ.get("OPENAI_API_KEY")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"OPENAI_API_KEY loaded ({len(token)} characters): {masked}")
else:
    print("OPENAI_API_KEY not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'OPENAI_API_KEY', and load_dotenv() ran without error.")

OPENAI_API_KEY loaded (164 characters): sk-p...UPEA


In [5]:
"""
Phase 2 triple extraction: multi-model comparison run.

Runs the same extraction prompt (phase2_extraction_prompt.md) against a
fixed sample of chunks, once per model, so outputs can be compared for
cost and accuracy before committing to one model for the full corpus.

Models compared:
    - gpt-5.6-terra       (OpenAI)
    - gpt-5.6-luna        (OpenAI)
    - gpt-5.5-pro          (OpenAI)
    - gemini-3.1-pro-preview  (Google)
    - gemini-3.6-flash        (Google)

Requires a .env file (not committed, not shared) with:
    OPENAI_API_KEY=...
    GEMINI_API_KEY=...

Usage:
    python run_phase2_model_comparison.py
"""

import os
import re
import json
import time
import random
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY not found. Add it to your .env file.")
if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY not found. Add it to your .env file.")

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

CHUNKS_PATH = r"C:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\PDF_PREPROCESSING_INTO_CHUNKS\chunks_all.parquet"
PROMPT_PATH = "phase2_extraction_prompt.md"
N_CHUNKS = 30
RANDOM_SEED = 42
OUTPUT_DIR = Path("model_comparison_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Pricing per 1M tokens (input, output), USD, as of Aug 2026.
# Used only to estimate comparative run cost, not billed automatically.
MODEL_PRICING = {
    "gpt-5.6-terra": (2.50, 15.00),
    "gpt-5.6-luna": (1.00, 6.00),
    "gpt-5.5-pro": (30.00, 180.00),
    "gemini-3.1-pro-preview": (2.00, 12.00),
    "gemini-3.6-flash": (1.50, 7.50),
}

OPENAI_MODELS = ["gpt-5.6-terra", "gpt-5.6-luna", "gpt-5.5-pro"]
GEMINI_MODELS = ["gemini-3.1-pro-preview", "gemini-3.6-flash"]


# ---------------------------------------------------------------------------
# Load prompt + chunk sample
# ---------------------------------------------------------------------------

def load_system_prompt(path: str) -> str:
    """Pull the system prompt block out of phase2_extraction_prompt.md.

    The markdown file mixes instructions and worked examples; the whole
    thing (minus the top-level H1) is used verbatim as the system prompt,
    since the few-shot examples are part of what makes the extraction work.
    """
    text = Path(path).read_text()
    # Drop the leading "# Phase 2: Triple Extraction Prompt" title line only.
    text = re.sub(r"^#\s+Phase 2.*\n", "", text, count=1)
    return text.strip()


def sample_chunks(path: str, n: int, seed: int) -> pd.DataFrame:
    df = pd.read_parquet(path)
    # Drop chunks with little or no extractable content (pure references,
    # empty rows, boilerplate) before sampling, so the 30-chunk comparison
    # set is actually representative of extraction difficulty.
    df = df[df["chunk_text"].str.strip().str.len() > 200].reset_index(drop=True)
    return df.sample(n=n, random_state=seed).reset_index(drop=True)


SYSTEM_PROMPT = load_system_prompt(PROMPT_PATH)
sample_df = sample_chunks(CHUNKS_PATH, N_CHUNKS, RANDOM_SEED)
sample_df.to_csv(OUTPUT_DIR / "sampled_30_chunks.csv", index=False)
print(f"Sampled {len(sample_df)} chunks -> {OUTPUT_DIR / 'sampled_30_chunks.csv'}")


def build_user_message(row: pd.Series) -> str:
    return (
        f"pmid: {row['doi']}\n"
        f"section: {row['section']}\n\n"
        f"chunk_text:\n{row['chunk_text']}"
    )


# ---------------------------------------------------------------------------
# JSON parsing helper (models sometimes wrap output in ```json fences)
# ---------------------------------------------------------------------------

def parse_triples(raw_text: str):
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_text.strip(), flags=re.MULTILINE)
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            parsed = [parsed]
        return parsed, None
    except json.JSONDecodeError as e:
        return None, f"JSON parse error: {e}"


# ---------------------------------------------------------------------------
# OpenAI runner (covers gpt-5.6-terra, gpt-5.6-luna, gpt-5.5-pro)
# ---------------------------------------------------------------------------

def run_openai_model(model_name: str, df: pd.DataFrame) -> pd.DataFrame:
    from openai import OpenAI

    client = OpenAI(api_key=OPENAI_API_KEY)
    rows = []

    for i, row in df.iterrows():
        user_msg = build_user_message(row)
        start = time.time()
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
                temperature=0,
            )
            elapsed = time.time() - start
            raw_output = response.choices[0].message.content
            usage = response.usage
            input_tokens = usage.prompt_tokens if usage else None
            output_tokens = usage.completion_tokens if usage else None
            triples, err = parse_triples(raw_output)

            rows.append({
                "model": model_name,
                "chunk_id": row["id"],
                "doi": row["doi"],
                "section": row["section"],
                "raw_output": raw_output,
                "n_triples": len(triples) if triples is not None else None,
                "parse_error": err,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "latency_sec": round(elapsed, 2),
            })
            print(f"  [{model_name}] {i+1}/{len(df)} ok "
                  f"({len(triples) if triples else 0} triples, {elapsed:.1f}s)")
        except Exception as e:
            rows.append({
                "model": model_name,
                "chunk_id": row["id"],
                "doi": row["doi"],
                "section": row["section"],
                "raw_output": None,
                "n_triples": None,
                "parse_error": f"API error: {e}",
                "input_tokens": None,
                "output_tokens": None,
                "latency_sec": None,
            })
            print(f"  [{model_name}] {i+1}/{len(df)} FAILED: {e}")

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Gemini runner (covers gemini-3.1-pro-preview, gemini-3.6-flash)
# ---------------------------------------------------------------------------

def run_gemini_model(model_name: str, df: pd.DataFrame) -> pd.DataFrame:
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=GEMINI_API_KEY)
    rows = []

    for i, row in df.iterrows():
        user_msg = build_user_message(row)
        start = time.time()
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=user_msg,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0,
                ),
            )
            elapsed = time.time() - start
            raw_output = response.text
            usage = getattr(response, "usage_metadata", None)
            input_tokens = usage.prompt_token_count if usage else None
            output_tokens = usage.candidates_token_count if usage else None
            triples, err = parse_triples(raw_output)

            rows.append({
                "model": model_name,
                "chunk_id": row["id"],
                "doi": row["doi"],
                "section": row["section"],
                "raw_output": raw_output,
                "n_triples": len(triples) if triples is not None else None,
                "parse_error": err,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "latency_sec": round(elapsed, 2),
            })
            print(f"  [{model_name}] {i+1}/{len(df)} ok "
                  f"({len(triples) if triples else 0} triples, {elapsed:.1f}s)")
        except Exception as e:
            rows.append({
                "model": model_name,
                "chunk_id": row["id"],
                "doi": row["doi"],
                "section": row["section"],
                "raw_output": None,
                "n_triples": None,
                "parse_error": f"API error: {e}",
                "input_tokens": None,
                "output_tokens": None,
                "latency_sec": None,
            })
            print(f"  [{model_name}] {i+1}/{len(df)} FAILED: {e}")

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Cost estimate
# ---------------------------------------------------------------------------

def add_cost_column(df: pd.DataFrame) -> pd.DataFrame:
    def _cost(r):
        if pd.isna(r["input_tokens"]) or pd.isna(r["output_tokens"]):
            return None
        in_price, out_price = MODEL_PRICING[r["model"]]
        return (r["input_tokens"] / 1_000_000 * in_price) + (r["output_tokens"] / 1_000_000 * out_price)
    df["est_cost_usd"] = df.apply(_cost, axis=1)
    return df


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    all_results = []

    for model_name in OPENAI_MODELS:
        print(f"\nRunning {model_name} on {len(sample_df)} chunks...")
        result_df = run_openai_model(model_name, sample_df)
        all_results.append(result_df)

    for model_name in GEMINI_MODELS:
        print(f"\nRunning {model_name} on {len(sample_df)} chunks...")
        result_df = run_gemini_model(model_name, sample_df)
        all_results.append(result_df)

    combined = pd.concat(all_results, ignore_index=True)
    combined = add_cost_column(combined)

    out_path = OUTPUT_DIR / "phase2_model_comparison_raw.xlsx"
    combined.to_excel(out_path, index=False)
    print(f"\nRaw comparison results -> {out_path}")

    # Summary: per-model rollup for a first-pass read before manual review.
    summary = (
        combined.groupby("model")
        .agg(
            chunks_run=("chunk_id", "count"),
            chunks_failed=("parse_error", lambda x: x.notna().sum()),
            avg_triples_per_chunk=("n_triples", "mean"),
            total_input_tokens=("input_tokens", "sum"),
            total_output_tokens=("output_tokens", "sum"),
            total_est_cost_usd=("est_cost_usd", "sum"),
            avg_latency_sec=("latency_sec", "mean"),
        )
        .reset_index()
    )
    summary["est_cost_per_1000_chunks_usd"] = (
        summary["total_est_cost_usd"] / len(sample_df) * 1000
    )
    summary_path = OUTPUT_DIR / "phase2_model_comparison_summary.xlsx"
    summary.to_excel(summary_path, index=False)
    print(f"Summary -> {summary_path}")
    print("\n" + summary.to_string(index=False))
    print(
        "\nNote: avg_triples_per_chunk and chunks_failed are volume/parse-success "
        "signals only, not accuracy. Accuracy still needs a manual read of "
        "phase2_model_comparison_raw.xlsx against the source chunks."
    )


if __name__ == "__main__":
    main()

Sampled 30 chunks -> model_comparison_output\sampled_30_chunks.csv

Running gpt-5.6-terra on 30 chunks...
  [gpt-5.6-terra] 1/30 FAILED: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************UPEA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
  [gpt-5.6-terra] 2/30 FAILED: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************UPEA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
  [gpt-5.6-terra] 3/30 FAI

  [gemini-3.1-pro-preview] 1/30 ok (18 triples, 115.3s)


  [gemini-3.1-pro-preview] 2/30 ok (25 triples, 272.1s)


  [gemini-3.1-pro-preview] 3/30 ok (27 triples, 121.7s)
  [gemini-3.1-pro-preview] 4/30 FAILED: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


  [gemini-3.1-pro-preview] 5/30 ok (0 triples, 49.7s)
  [gemini-3.1-pro-preview] 6/30 FAILED: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


...

  [gemini-3.6-flash] 27/30 ok (8 triples, 18.9s)


  [gemini-3.6-flash] 28/30 ok (0 triples, 8.5s)


  [gemini-3.6-flash] 29/30 ok (4 triples, 20.2s)


C:\Users\olagunju\AppData\Local\Temp\ipykernel_59372\688702163.py:277: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(all_results, ignore_index=True)


  [gemini-3.6-flash] 30/30 ok (8 triples, 23.7s)

Raw comparison results -> model_comparison_output\phase2_model_comparison_raw.xlsx
Summary -> model_comparison_output\phase2_model_comparison_summary.xlsx

                 model  chunks_run  chunks_failed  avg_triples_per_chunk  total_input_tokens  total_output_tokens  total_est_cost_usd  avg_latency_sec  est_cost_per_1000_chunks_usd
gemini-3.1-pro-preview          30              7              10.826087            131266.0              49904.0            0.861380        98.580435                     28.712667
      gemini-3.6-flash          30              1               5.896552            163718.0              34491.0            0.504259        21.634828                     16.808650
           gpt-5.5-pro          30             30                    NaN                 0.0                  0.0            0.000000              NaN                      0.000000
          gpt-5.6-luna          30             30                    N